# S6E9 | Data Contracts, Duplicate Checks and ID-Aware Drift

Before training an EV purchase model, check what the inputs actually support.
This CPU-only notebook audits **all official train/test rows**, keeps identifiers
separate from predictors, and returns reusable aggregate diagnostics. It trains
no model, accesses no hidden labels and creates no prediction submission.

The central question is narrow: **does a train/test difference come from an ID,
from an actual predictor, or from a data-contract problem?** Small marginal
distances are not proof of joint-distribution stability or leaderboard generalization.

**Data:** Kaggle Playground Series S6E9, official files downloaded on 2026-09-07.
The competition data are inspired by another dataset and are not a representative
survey of real EV purchasers. Descriptive patterns must not be read as causal
effects or business forecasts.

**Reproducibility:** attach the competition input, then run all cells. No external
datasets, credentials, downloads or helper notebooks are required. File hashes
and installed package versions are printed below. Raw records are not displayed
or exported. Results are recomputed, not hard-coded.

In [ ]:
import os
import sys
import json
import time
import hashlib
import platform
import resource
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

START = time.perf_counter()
TARGET, ID = "Will_Buy_EV", "id"
override = os.environ.get("EV_DATA_DIR")
roots = [Path(override)] if override else [
    Path("/kaggle/input/competitions/playground-series-s6e9"),
    Path("/kaggle/input/playground-series-s6e9"),
]
valid = [p for p in roots if all((p / name).is_file() for name in
         ("train.csv", "test.csv", "sample_submission.csv"))]
if len(valid) != 1:
    raise RuntimeError("Attach exactly one official S6E9 input, or set EV_DATA_DIR locally.")
DATA = valid[0]
OUTPUT = Path("ev_a_aggregate_audit")
OUTPUT.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False})

def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

manifest = {"observed_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "packages": {p: version(p) for p in ("pandas", "numpy", "scipy", "matplotlib")},
    "files": {name: {"bytes": (DATA/name).stat().st_size, "sha256": digest(DATA/name)}
              for name in ("train.csv", "test.csv", "sample_submission.csv")}}
print(json.dumps(manifest, indent=2))

## 1. Test the input contract before interpreting data

The contract checks that training labels are explicit `Yes`/`No`, test labels are
absent, IDs are non-null and unique, train/test IDs do not overlap, and the sample
file has exactly the test ID order. It does not treat the sample probabilities as
ground truth. Dtype inference alone is not a semantic feature specification.

The helpers below use pandas and SciPy. Numeric KS distance is the largest
empirical-CDF gap, on a 0-to-1 scale. Categorical total variation is half the sum
of absolute category-frequency differences, also 0 to 1; missing categories are
handled separately from literal category names. These are effect-size diagnostics,
not hypothesis-test p-values.

In [ ]:
"""Original aggregate-only audit helpers for S6E9. No network or data export."""

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp


def check_contract(train, test, sample, target="Will_Buy_EV", identifier="id"):
    if any(df.empty for df in (train, test, sample)):
        raise ValueError("Inputs must not be empty")
    if any(not df.columns.is_unique for df in (train, test, sample)):
        raise ValueError("Duplicate column names")
    if target not in train or target in test:
        raise ValueError("Target must occur only in the training input")
    features = [c for c in train if c not in (target, identifier)]
    if not features or set(test.columns) != set(features + [identifier]):
        raise ValueError("Training and test feature schemas differ")
    for df in (train, test, sample):
        if identifier not in df or df[identifier].isna().any() or not df[identifier].is_unique:
            raise ValueError("IDs must be present, non-null and unique")
    if len(sample) != len(test) or not np.array_equal(sample[identifier], test[identifier]):
        raise ValueError("Sample IDs must match test IDs in row order")
    if set(sample.columns) != {identifier, target}:
        raise ValueError("Unexpected sample submission schema")
    if train[identifier].isin(test[identifier]).any():
        raise ValueError("Train/test IDs overlap")
    if train[target].isna().any() or set(train[target].unique()) != {"No", "Yes"}:
        raise ValueError("Expected non-null Yes/No labels; inspect a changed data version")
    return features


def duplicate_audit(train, test, features, target="Will_Buy_EV"):
    # MultiIndex compares full tuples, without relying on a row-hash collision assumption.
    train_key = pd.MultiIndex.from_frame(train[features])
    test_key = pd.MultiIndex.from_frame(test[features])
    duplicated = train.duplicated(features, keep=False)
    conflicts = 0
    if duplicated.any():
        counts = train.loc[duplicated].groupby(features, dropna=False)[target].nunique()
        conflicts = int((counts > 1).sum())
    return {
        "train_extra_duplicate_feature_rows": int(train_key.duplicated().sum()),
        "test_extra_duplicate_feature_rows": int(test_key.duplicated().sum()),
        "test_rows_matching_any_train_feature_tuple": int(test_key.isin(train_key).sum()),
        "train_feature_groups_with_conflicting_labels": conflicts,
    }


def numeric_shift(train, test, columns):
    rows = []
    for col in columns:
        a = train[col].to_numpy(dtype=float, na_value=np.nan)
        b = test[col].to_numpy(dtype=float, na_value=np.nan)
        x, y = a[np.isfinite(a)], b[np.isfinite(b)]
        stat = float(ks_2samp(x, y, method="asymp").statistic) if len(x) and len(y) else np.nan
        outside = float(((y < x.min()) | (y > x.max())).mean()) if len(x) and len(y) else np.nan
        rows.append({"feature": col, "ks_distance": stat,
                     "train_nonfinite_rate": float(1 - len(x) / len(a)),
                     "test_nonfinite_rate": float(1 - len(y) / len(b)),
                     "test_outside_train_range_rate": outside})
    return pd.DataFrame(rows).set_index("feature")


def categorical_shift(train, test, columns):
    rows = []
    for col in columns:
        a, b = train[col].astype("string"), test[col].astype("string")
        missing = "__AUDIT_MISSING__"
        while a.eq(missing).any() or b.eq(missing).any():
            missing += "_"
        a, b = a.fillna(missing), b.fillna(missing)
        p, q = a.value_counts(normalize=True), b.value_counts(normalize=True)
        support = p.index.union(q.index)
        tv = float((p.reindex(support, fill_value=0) - q.reindex(support, fill_value=0)).abs().sum() / 2)
        rows.append({"feature": col, "total_variation": tv,
                     "test_unseen_category_rate": float((~b.isin(p.index)).mean()),
                     "train_levels_including_missing": len(p),
                     "test_levels_including_missing": len(q)})
    return pd.DataFrame(rows).set_index("feature")

### Small known-answer controls (synthetic test fixtures)

These tiny fixtures are not competition records. They exercise an exact
overlap, conflicting labels, an ID-only separation and an unseen category.
They test the implementation; they do not simulate the competition's difficulty.

In [ ]:
toy_train = pd.DataFrame({"id": [0, 1, 2, 3], "x": [1, 1, 2, 3],
    "c": ["a", "a", "b", "c"], TARGET: ["No", "Yes", "No", "Yes"]})
toy_test = pd.DataFrame({"id": [4, 5], "x": [1, 4], "c": ["a", "d"]})
toy_sample = pd.DataFrame({"id": [4, 5], TARGET: [.5, .5]})
assert check_contract(toy_train, toy_test, toy_sample) == ["x", "c"]
toy_duplicate = duplicate_audit(toy_train, toy_test, ["x", "c"])
assert toy_duplicate["train_feature_groups_with_conflicting_labels"] == 1
assert toy_duplicate["test_rows_matching_any_train_feature_tuple"] == 1
assert numeric_shift(toy_train, toy_test, [ID]).loc[ID, "ks_distance"] == 1.0
assert categorical_shift(toy_train, toy_test, ["c"]).loc["c", "test_unseen_category_rate"] == .5
try:
    check_contract(toy_train, toy_test, toy_sample.iloc[::-1])
except ValueError:
    pass
else:
    raise AssertionError("Reversed sample ID order was not rejected")
print("Known-answer controls: PASS")
del toy_train, toy_test, toy_sample, toy_duplicate

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
features = check_contract(train, test, sample)
numeric = train[features].select_dtypes(include="number").columns.tolist()
categorical = [c for c in features if c not in numeric]
y = train[TARGET].map({"No": 0, "Yes": 1})
schema = pd.DataFrame({"train_dtype": train[features].dtypes.astype(str),
    "test_dtype": test[features].dtypes.astype(str),
    "train_missing": train[features].isna().sum(),
    "test_missing": test[features].isna().sum(),
    "train_unique": train[features].nunique(dropna=False),
    "test_unique": test[features].nunique(dropna=False)})
display(pd.DataFrame({"rows": [len(train), len(test)],
                     "predictors": [len(features), len(features)]}, index=["train", "test"]))
display(schema)
print("Input contract: PASS")

## 2. Target balance and exact feature overlap

Duplicate checks exclude both ID and target. An "extra duplicate row" counts
all occurrences after the first identical feature tuple. A conflicting-label
group is an identical training-feature tuple associated with both labels.
The overlap count measures test rows matching any complete train-feature tuple.

Zero exact matches does **not** exclude near-duplicates, shared latent groups or
relationships to the source dataset. No source dataset is loaded in this audit.

In [ ]:
duplicates = duplicate_audit(train, test, features)
display(pd.Series(duplicates, name="count").to_frame())
target_counts = train[TARGET].value_counts().reindex(["No", "Yes"], fill_value=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
axes[0].bar(target_counts.index, target_counts.values, color=["#52616B", "#16817A"])
axes[0].set(title="Training labels (full dataset)", ylabel="Rows")
for i, count in enumerate(target_counts):
    axes[0].text(i, count, f"{count:,} ({count/len(train):.2%})", ha="center", va="bottom")
axes[0].set_ylim(0, target_counts.max()*1.18)
miss = [int(train[features].isna().sum().sum()), int(test[features].isna().sum().sum())]
axes[1].bar(["train", "test"], miss, color=["#52616B", "#D1603D"])
axes[1].set(title="Missing predictor cells (full dataset)", ylabel="Cells")
axes[1].set_ylim(0, max(1, max(miss)*1.2))
for i, count in enumerate(miss):
    axes[1].text(i, count, f"{count:,}", ha="center", va="bottom")
fig.savefig(OUTPUT / "01_balance_and_missing.png", bbox_inches="tight")
plt.show()
display(Markdown(f"The observed training positive rate is **{y.mean():.3%}**. "
    "Accuracy alone can hide poor positive-class performance; the competition uses ROC-AUC. "
    "This audit reports no validation AUC or leaderboard score."))

## 3. The identifier is a negative control, not a predictor

Compute the same marginal-distance statistic for the ID and real predictors,
but report them separately. A perfectly separating ID can dominate a train-vs-test
diagnostic without indicating any shift in useful predictor information.
This does not prove an ID could never correlate with a target; it means such a
relationship would require separate leakage and validation scrutiny.

In [ ]:
num_report = numeric_shift(train, test, numeric)
id_report = numeric_shift(train, test, [ID])
display(num_report.sort_values("ks_distance", ascending=False).round(6))
display(id_report.round(6))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [3, 1]}, layout="constrained")
ordered = num_report["ks_distance"].sort_values()
axes[0].barh(ordered.index, ordered.values, color="#16817A")
axes[0].set(title="Predictors: marginal KS distance", xlabel="KS (separate zoomed scale)")
axes[0].set_xlim(0, max(.005, ordered.max()*1.25))
axes[1].bar(["id"], [id_report.loc[ID, "ks_distance"]], color="#D1603D")
axes[1].set(title="Identifier control", ylabel="KS (0 to 1)", ylim=(0, 1.08))
fig.savefig(OUTPUT / "02_id_control.png", bbox_inches="tight")
plt.show()
display(Markdown(f"Maximum numeric predictor KS: **{num_report.ks_distance.max():.6f}**; "
    f"identifier KS: **{id_report.loc[ID, 'ks_distance']:.6f}**. "
    "The panels deliberately use different axis scales. Do not include ID when summarizing predictor drift."))

## 4. Category support and the size of marginal differences

Unseen-category rate asks a different question from frequency shift: does a test
row contain a level absent from training? Total variation can be small even if
a rare unseen level exists. The following calculations include every row.

In [ ]:
cat_report = categorical_shift(train, test, categorical)
display(cat_report.sort_values("total_variation", ascending=False).round(6))
ordered = cat_report["total_variation"].sort_values()
fig, ax = plt.subplots(figsize=(9, 3.8), layout="constrained")
ax.barh(ordered.index, ordered.values, color="#52616B")
ax.set(title="Categorical predictors: total variation", xlabel="TV distance (zoomed scale)")
ax.set_xlim(0, max(.005, ordered.max()*1.25))
fig.savefig(OUTPUT / "03_category_shift.png", bbox_inches="tight")
plt.show()

## 5. Quantile views: compare distributions, not just a scalar

These curves use 501 quantiles computed from all finite rows, not an undocumented
row sample. Quantile interpolation and plot resolution can hide small tail
differences, so the full-data KS and out-of-range rates above remain relevant.
Near-overlap is not evidence that feature interactions or conditional label
distributions match: test labels are unavailable.

In [ ]:
view_columns = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Environmental_Concern_Level"]
probabilities = np.linspace(0, 1, 501)
fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")
for ax, col in zip(axes.flat, view_columns):
    for frame, label, color, style in [(train, "train", "#16817A", "-"), (test, "test", "#D1603D", "--")]:
        vals = frame[col].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        ax.plot(np.quantile(vals, probabilities), probabilities, label=label, color=color, linestyle=style)
    ax.set(title=col, ylabel="Quantile probability", xlabel="Feature value")
    ax.legend(loc="best")
fig.savefig(OUTPUT / "04_quantile_views.png", bbox_inches="tight")
plt.show()

## 6. Reusable decision record and limitations

The aggregate report records what was checked, not a claim that a dataset is
"safe". Missingness, exact duplicates, univariate distances and input consistency
are only part of validation design. No pass/fail threshold for drift is tuned to
these observations, and no model improvement is claimed.

**Next experiment:** investigate a justified validation design, including
near-duplicate or latent-group structure only if there is actual evidence for it.
Do not invent groups because exact-duplicate counts are zero. Do not tune
feature engineering from this test-set audit without recording that exposure.

In [ ]:
summary = {"rows": {"train": len(train), "test": len(test)},
    "predictors": len(features), "numeric_predictors": len(numeric),
    "categorical_predictors": len(categorical), "positive_rate_train": float(y.mean()),
    "missing_predictor_cells": {"train": int(train[features].isna().sum().sum()),
                                "test": int(test[features].isna().sum().sum())},
    "duplicates": duplicates,
    "max_numeric_predictor_ks": float(num_report.ks_distance.max()),
    "identifier_ks": float(id_report.loc[ID, "ks_distance"]),
    "max_categorical_tv": float(cat_report.total_variation.max()),
    "max_unseen_category_rate": float(cat_report.test_unseen_category_rate.max()),
    "audit_seconds": round(time.perf_counter()-START, 2),
    "process_peak_rss_mib": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss /
        (1024**2 if sys.platform == "darwin" else 1024), 2),
    "runtime_scope": "Audit timer excludes kernel startup and initial imports; RSS is process peak.",
    "manifest": manifest,
    "validation_auc": None, "leaderboard_score": None, "submission_created": False}
for name, table in [("schema", schema), ("numeric_shift", num_report), ("categorical_shift", cat_report)]:
    table.to_csv(OUTPUT / f"{name}.csv")
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False))
print(json.dumps({k:v for k,v in summary.items() if k!='manifest'}, indent=2))
print("Only aggregate diagnostics and charts were written. No raw rows or predictions were exported.")

## Sources and attribution

- [Official competition](https://www.kaggle.com/competitions/playground-series-s6e9),
  Yao Yan, Walter Reade and Elizabeth Park, Kaggle, 2026.
- [Official data description](https://www.kaggle.com/competitions/playground-series-s6e9/data)
  and [competition rules](https://www.kaggle.com/competitions/playground-series-s6e9/rules).
  Obtain data through Kaggle after accepting the rules; this notebook does not redistribute it.
- Related reading: Aryan Kaisth,
  [Signal that Matters (Complete EDA)](https://www.kaggle.com/code/aryankaisth/signal-that-matters-complete-eda),
  version 14, observed 2026-09-07, Apache 2.0. It examines feature/target patterns and
  already notes limited marginal drift. This notebook does not claim that observation
  as new; its focus is executable contracts, exact tuple overlap and an explicit ID control.
  No code from that notebook is copied or executed here.
- pandas supplies exact tuple operations; SciPy supplies the two-sample KS statistic.

Original analysis code was prepared with AI assistance and tested before publication.
No generalization, causal, business, or leaderboard-performance claim is made.